# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

#### Preparación

In [1]:
# Librerías
import math
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

TOLERANCIA = 0.00000001

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [2]:
# 1. Recuperar los archivos TXT
carpeta = Path(r"C:\Users\Juan\Downloads\txt-files.tar\txt-files\cache\epub")

archivos_txt = [
    ruta
    for ruta in carpeta.rglob("*")
    if ruta.is_file() and ruta.suffix.lower() == ".txt"
]

print("Archivos TXT encontrados:", len(archivos_txt))
print("Primeros 10 archivos TXT encontrados:")
for ruta in archivos_txt[:10]:
    print(ruta.name)


def obtener_id(ruta):
    nombre = ruta.stem.lower()
    coincidencia = re.match(r"(?:pg)?(\d+)(?:[-_].*)?$", nombre)
    if coincidencia:
        return coincidencia.group(1)
    return None


libros = {}
sin_id = []
for ruta in sorted(archivos_txt):
    if ruta.stat().st_size == 0:
        continue
    id_libro = obtener_id(ruta)
    if id_libro is None:
        sin_id.append(ruta)
    elif id_libro not in libros:
        libros[id_libro] = ruta

print("Libros con ID único:", len(libros))
print("TXT sin ID reconocido:", len(sin_id))


patron = r"[a-z0-9]+(?:'[a-z]+)*"


def preprocesamiento(texto):
    texto = texto.lower()
    texto = texto.replace("’", "'")
    palabras = re.findall(patron, texto)
    return palabras


def crear_subconjunto(cantidad):
    corpus = {}
    seleccionados = list(libros.items())[:cantidad]
    for id_libro, ruta in seleccionados:
        texto = ruta.read_text(encoding="utf-8", errors="replace")
        corpus[id_libro] = preprocesamiento(texto)
    return corpus


def obtener_titulo(texto, ruta):
    coincidencia = re.search(r"^Title: (.+)$", texto, re.MULTILINE)
    if coincidencia:
        return coincidencia.group(1).strip()
    return ruta.stem

# 1. Vocabulario
def construir_vocabulario(documentos):
    palabras = set()
    for documento in documentos:
        palabras.update(documento)
    return sorted(palabras)

# 1. Tabla y estadísticas
def revisar_corpus(identificadores, titulos, documentos):
    filas = []
    longitudes = []
    for posicion, tokens in enumerate(documentos):
        longitudes.append(len(tokens))
        filas.append({
            'Identificador': identificadores[posicion],
            'Titulo': titulos[posicion],
            'Terminos unicos': len(set(tokens)),
        })
    resumen = {
        'Documentos': len(documentos),
        'Vocabulario': len(construir_vocabulario(documentos)),
        'Longitud minima': min(longitudes),
        'Longitud media': sum(longitudes) / len(longitudes),
        'Longitud maxima': max(longitudes),
    }
    return pd.DataFrame(filas), resumen

# 1. Resultado
CANTIDAD = 50
corpus = crear_subconjunto(CANTIDAD)
if len(corpus) < 5:
    raise ValueError("Se necesitan al menos 5 libros para mostrar el top-5. Revisa carpeta y CANTIDAD.")

identificadores = list(corpus.keys())
documentos = list(corpus.values())
titulos = []
textos_originales = []
for id_libro in identificadores:
    ruta = libros[id_libro]
    texto = ruta.read_text(encoding="utf-8", errors="replace")
    textos_originales.append(texto)
    titulos.append(obtener_titulo(texto, ruta))

tabla_corpus, resumen_corpus = revisar_corpus(identificadores, titulos, documentos)
display(tabla_corpus)
display(pd.DataFrame([resumen_corpus]))

Archivos TXT encontrados: 78183
Primeros 10 archivos TXT encontrados:
pg1.txt
pg10.txt
pg100.txt
pg1000.txt
pg10000.txt
pg10001.txt
pg10002.txt
pg10003.txt
pg10004.txt
pg10005.txt
Libros con ID único: 78183
TXT sin ID reconocido: 0


Identificador,Titulo,Terminos unicos
1,The Declaration of Independence of the United States of America,1216
10,The King James Version of the Bible,13283
100,The Complete Works of William Shakespeare,27515
1000,La Divina Commedia di Dante: Complete,14090
10000,The Magna Carta,1883
10001,Apocolocyntosis,2134
10002,The House on the Borderland,5372
10003,"My First Years as a Frenchwoman, 1876-1879",6391
10004,The Warriors,6908
10005,A Voyage to the Moon,7959


Documentos,Vocabulario,Longitud minima,Longitud media,Longitud maxima
50,76426,4920,82716.3,971907


### 2. Implementación de Funciones

Programe funciones para obtener:

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

1. TF de cada término en cada documento.

$TF(t,d)=f(t,d)$

In [3]:
# 2.1. TF
def calcular_tf(tokens):
    return dict(Counter(tokens))

# 2.1. Resultado
frecuencias_tf = calcular_tf(documentos[0])
display(pd.DataFrame(frecuencias_tf.items(), columns=["Termino", "TF"]).head(10))

Termino,TF
the,306
project,91
gutenberg,100
ebook,14
of,236
declaration,11
independence,8
united,25
states,34
america,7


2. DF de cada término en **todo el corpus**.

In [4]:
# 2.2. DF
def calcular_df(documentos):
    frecuencias = Counter()
    for documento in documentos:
        frecuencias.update(set(documento))
    return dict(frecuencias)

# 2.2. Resultado
frecuencias_df = calcular_df(documentos)
display(pd.DataFrame(frecuencias_df.items(), columns=["Termino", "DF"]).head(10))

Termino,DF
mock,13
human,43
charge,50
sharing,50
agreed,50
such,50
evils,13
reported,50
trademark,50
reasonable,50


3. IDF de cada término.

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [5]:
# 2.3. IDF
def calcular_idf(frecuencias_df, numero_documentos):
    valores = {}
    for palabra, frecuencia in frecuencias_df.items():
        valores[palabra] = math.log(numero_documentos / frecuencia) + 1
    return valores

# 2.3. Resultado
valores_idf = calcular_idf(frecuencias_df, len(documentos))
display(pd.DataFrame(valores_idf.items(), columns=["Termino", "IDF"]).head(10))

Termino,IDF
mock,2.347074
human,1.150823
charge,1.000000
sharing,1.000000
agreed,1.000000
such,1.000000
evils,2.347074
reported,1.000000
trademark,1.000000
reasonable,1.000000


4. Los seis pesos TF-IDF correspondientes.

$
w_{t,d}=TF(t,d)\times IDF(t)
$

In [6]:
# 2.4. Peso TF-IDF
def calcular_peso(tf, idf):
    return tf * idf

# 2.4. Resultado: seis valores con la fórmula indicada en el enunciado
pesos = []
for posicion in range(2):
    frecuencias = calcular_tf(documentos[posicion])
    for palabra in ["love", "sea", "king"]:
        tf = frecuencias.get(palabra, 0)
        idf = valores_idf.get(palabra, 0)
        pesos.append([identificadores[posicion], palabra, tf, idf, calcular_peso(tf, idf)])
display(pd.DataFrame(pesos, columns=["Documento", "Termino", "TF", "IDF", "TF-IDF"]))

Documento,Termino,TF,IDF,TF-IDF
1,love,0,1.174353,0.000000
1,sea,0,1.150823,0.000000
1,king,1,1.301105,1.301105
10,love,310,1.174353,364.049550
10,sea,400,1.150823,460.329156
10,king,2264,1.301105,2945.701930


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.

In [7]:
# 3.1. Resultado: vocabulario ordenado
vocabulario = construir_vocabulario(documentos)
print("Palabras:", len(vocabulario))
print("Primeras 20:", vocabulario[:20])

Palabras: 76426
Primeras 20: ['0', '00', '000', '004', '01', '02', '044', '05', '06', '07', '07042', '09', '1', '10', '100', '1000', '10000', '10001', '10002', '10003']


2. Contar las frecuencias TF.

In [8]:
# 3.2. Matriz TF
def construir_matriz_tf(documentos, vocabulario):
    matriz = []
    for documento in documentos:
        frecuencias = calcular_tf(documento)
        fila = []
        for palabra in vocabulario:
            fila.append(frecuencias.get(palabra, 0))
        matriz.append(fila)
    return np.array(matriz, dtype=float)

# 3.2. Resultado
matriz_tf = construir_matriz_tf(documentos, vocabulario)
display(pd.DataFrame(matriz_tf[:5, :8], index=identificadores[:5], columns=vocabulario[:8]))

,0,00,000,004,01,02,044,05
1,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
10,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
100,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1000,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
10000,4.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


3. Calcular DF contando cada documento una sola vez por término.

In [9]:
# 3.3. Resultado: DF de todo el corpus
frecuencias_df = calcular_df(documentos)
display(pd.DataFrame({"Termino": vocabulario[:8], "DF": [frecuencias_df[p] for p in vocabulario[:8]]}))

Termino,DF
0,7
00,15
000,50
004,1
01,1
02,1
044,1
05,1


4. Calcular IDF con la función de la sección anterior.

In [10]:
# 3.4. Resultado: IDF del corpus
valores_idf = calcular_idf(frecuencias_df, len(documentos))
display(pd.DataFrame({"Termino": vocabulario[:8], "IDF": [valores_idf[p] for p in vocabulario[:8]]}))

Termino,IDF
0,2.966113
00,2.203973
000,1.000000
004,4.912023
01,4.912023
02,4.912023
044,4.912023
05,4.912023


5. Obtener la matriz documento-término TF-IDF.

In [11]:
# 3.5. Matriz TF-IDF
def construir_matriz_tfidf(matriz_tf, vocabulario, valores_idf):
    matriz = matriz_tf.copy()
    for columna, palabra in enumerate(vocabulario):
        matriz[:, columna] = matriz_tf[:, columna] * valores_idf[palabra]
    return matriz

# 3.5. Resultado
matriz_tfidf = construir_matriz_tfidf(matriz_tf, vocabulario, valores_idf)
print("Dimensiones:", matriz_tfidf.shape)
display(pd.DataFrame(matriz_tfidf[:5, :8], index=identificadores[:5], columns=vocabulario[:8]))

Dimensiones: (50, 76426)


,0,00,000,004,01,02,044,05
1,0.000000,0.0,2.0,0.0,0.0,0.0,0.0,0.0
10,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0
100,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1000,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0
10000,11.864451,0.0,1.0,0.0,0.0,0.0,0.0,0.0


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [12]:
# 4. Conservar los mismos tokens
def conservar_tokens(tokens):
    return tokens

# 4. Comparar con scikit-learn
def verificar_tfidf(documentos, vocabulario, matriz_manual):
    vectorizador = TfidfVectorizer(
        analyzer=conservar_tokens,
        vocabulary=vocabulario,
        use_idf=True,
        smooth_idf=False,
        sublinear_tf=False,
        norm=None,
    )
    matriz_libreria = vectorizador.fit_transform(documentos).toarray()
    diferencia = np.max(np.abs(matriz_manual - matriz_libreria))
    return vectorizador, matriz_libreria, diferencia

# 4. Resultado
vectorizador, matriz_libreria, diferencia = verificar_tfidf(documentos, vocabulario, matriz_tfidf)
print("Máxima diferencia absoluta:", diferencia)
assert np.allclose(matriz_tfidf, matriz_libreria, atol=TOLERANCIA, rtol=0)
print("Las matrices coinciden.")

Máxima diferencia absoluta: 0.0
Las matrices coinciden.


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.

In [13]:
# 5.1. Resultado: cuatro consultas y su intención
consultas = ["love marriage family", "sea ship whale", "king murder ghost", "god heaven angels"]
intenciones = ["Relaciones amorosas y familiares", "Viajes por mar y ballenas", "Conflictos de reyes, asesinatos y fantasmas", "Religión, cielo y ángeles"]
display(pd.DataFrame({"Consulta": consultas, "Informacion esperada": intenciones}))

Consulta,Informacion esperada
love marriage family,Relaciones amorosas y familiares
sea ship whale,Viajes por mar y ballenas
king murder ghost,"Conflictos de reyes, asesinatos y fantasmas"
god heaven angels,"Religión, cielo y ángeles"


2. Apliquen el mismo preprocesamiento de los documentos.

In [14]:
# 5.2. Resultado: mismo preprocesamiento
consultas_procesadas = []
for consulta in consultas:
    consultas_procesadas.append(preprocesamiento(consulta))
display(pd.DataFrame({"Consulta": consultas, "Tokens": consultas_procesadas}))

Consulta,Tokens
love marriage family,"[love, marriage, family]"
sea ship whale,"[sea, ship, whale]"
king murder ghost,"[king, murder, ghost]"
god heaven angels,"[god, heaven, angels]"


3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.

In [15]:
# 5.3. Vector de consulta
def vector_consulta(tokens, vocabulario, valores_idf):
    frecuencias = calcular_tf(tokens)
    vector = []
    for palabra in vocabulario:
        peso = frecuencias.get(palabra, 0) * valores_idf[palabra]
        vector.append(peso)
    return vector

# 5.3. Resultado
vectores_consultas = []
for tokens in consultas_procesadas:
    vectores_consultas.append(vector_consulta(tokens, vocabulario, valores_idf))


print("Forma de los vectores de consulta:", np.array(vectores_consultas).shape)

Forma de los vectores de consulta: (4, 76426)


4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

In [16]:
# 5.4. Resultado
consultas_libreria = vectorizador.transform(consultas_procesadas).toarray()
assert np.allclose(vectores_consultas, consultas_libreria, atol=TOLERANCIA, rtol=0)
print("Los cuatro vectores coinciden con transform.")
print("Se conserva el IDF de los documentos.")

Los cuatro vectores coinciden con transform.
Se conserva el IDF de los documentos.


**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

In [17]:
# 5. Similitud coseno
def producto_punto(vector_a, vector_b):
    total = 0
    for a, b in zip(vector_a, vector_b):
        total += a * b
    return total

# 5. Similitud coseno
def calcular_norma(vector):
    return math.sqrt(producto_punto(vector, vector))

# 5. Similitud coseno
def calcular_coseno(vector_a, vector_b):
    norma_a = calcular_norma(vector_a)
    norma_b = calcular_norma(vector_b)
    if norma_a == 0 or norma_b == 0:
        return 0.0
    return producto_punto(vector_a, vector_b) / (norma_a * norma_b)

# 5. Similitud coseno
def similitudes_documentos(consulta, matriz_documentos):
    similitudes = []
    for documento in matriz_documentos:
        similitudes.append(calcular_coseno(consulta, documento))
    return similitudes

# 5. Resultado: comparar similitudes
similitudes_manuales = []
for vector in vectores_consultas:
    similitudes_manuales.append(similitudes_documentos(vector, matriz_tfidf))
similitudes_libreria = cosine_similarity(consultas_libreria, matriz_libreria)
assert np.allclose(similitudes_manuales, similitudes_libreria, atol=TOLERANCIA, rtol=0)
print("Máxima diferencia:", np.max(np.abs(np.array(similitudes_manuales) - similitudes_libreria)))

Máxima diferencia: 5.273559366969494e-16


Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [18]:
# 5. Similitud coseno
def detalle_coseno(consulta, documento, vocabulario):
    filas = []
    for posicion, palabra in enumerate(vocabulario):
        if consulta[posicion] > 0 and documento[posicion] > 0:
            filas.append({
                'Termino': palabra,
                'Peso consulta': consulta[posicion],
                'Peso documento': documento[posicion],
                'Producto parcial': consulta[posicion] * documento[posicion],
            })
    tabla = pd.DataFrame(filas, columns=[
        'Termino', 'Peso consulta', 'Peso documento', 'Producto parcial'
    ])
    resumen = {
        'Producto punto': producto_punto(consulta, documento),
        'Norma consulta': calcular_norma(consulta),
        'Norma documento': calcular_norma(documento),
        'Similitud': calcular_coseno(consulta, documento),
    }
    return tabla, resumen

detalle, resumen = detalle_coseno(vectores_consultas[0], matriz_tfidf[0], vocabulario)
display(detalle)
display(pd.DataFrame([resumen]))

Termino,Peso consulta,Peso documento,Producto parcial


Producto punto,Norma consulta,Norma documento,Similitud
0.0,2.236064,539.887872,0.0


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [19]:
# 6. Ranking
def construir_ranking(identificadores, titulos, textos_originales, puntuaciones):
    filas = []
    for posicion in range(len(identificadores)):
        fragmento = ' '.join(textos_originales[posicion].split())[:250]
        filas.append({
            'Identificador': identificadores[posicion],
            'Titulo': titulos[posicion],
            'Puntuacion': puntuaciones[posicion],
            'Fragmento': fragmento,
        })
    tabla = pd.DataFrame(filas)
    tabla = tabla.sort_values('Puntuacion', ascending=False, kind='stable')
    tabla = tabla.head(5).reset_index(drop=True)
    tabla.insert(0, 'Posicion', range(1, len(tabla) + 1))
    return tabla

# 6. Resultado: top-5 de cada consulta
for posicion, consulta in enumerate(consultas):
    print("Consulta:", consulta)
    ranking = construir_ranking(identificadores, titulos, textos_originales, similitudes_manuales[posicion])
    display(ranking)

Consulta: love marriage family


Posicion,Identificador,Titulo,Puntuacion,Fragmento
1,10039,"The Works of Aphra Behn, Volume III",0.027213,"The Project Gutenberg eBook of The Works of Aphra Behn, Volume III This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it awa"
2,10006,La Fiammetta,0.023554,"The Project Gutenberg eBook of La Fiammetta This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under th"
3,100,The Complete Works of William Shakespeare,0.020276,"The Project Gutenberg eBook of The Complete Works of William Shakespeare This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give"
4,1003,"Divine Comedy, Longfellow's Translation, Paradise",0.015139,"The Project Gutenberg eBook of Divine Comedy, Longfellow's Translation, Paradise This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy i"
5,10041,The Rivet in Grandfather's Neck: A Comedy of Limitations,0.012486,The Project Gutenberg eBook of The Rivet in Grandfather's Neck: A Comedy of Limitations This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may


Consulta: sea ship whale


Posicion,Identificador,Titulo,Puntuacion,Fragmento
1,10015,"Punchinello, Volume 1, No. 19, August 6, 1870",0.019544,"The Project Gutenberg eBook of Punchinello, Volume 1, No. 19, August 6, 1870 This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, g"
2,10040,Aleutian Indian and English Dictionary,0.017098,"The Project Gutenberg eBook of Aleutian Indian and English Dictionary This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it"
3,10008,The Mystery,0.013706,"The Project Gutenberg eBook of The Mystery This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the"
4,10024,Beneath the Banner: Being Narratives of Noble Lives and Brave Deeds,0.005968,The Project Gutenberg eBook of Beneath the Banner: Being Narratives of Noble Lives and Brave Deeds This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoev
5,10012,The Mountains of California,0.003335,"The Project Gutenberg eBook of The Mountains of California This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-"


Consulta: king murder ghost


Posicion,Identificador,Titulo,Puntuacion,Fragmento
1,100,The Complete Works of William Shakespeare,0.026302,"The Project Gutenberg eBook of The Complete Works of William Shakespeare This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give"
2,10,The King James Version of the Bible,0.014158,"The Project Gutenberg eBook of The King James Version of the Bible This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it awa"
3,10039,"The Works of Aphra Behn, Volume III",0.009119,"The Project Gutenberg eBook of The Works of Aphra Behn, Volume III This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it awa"
4,10000,The Magna Carta,0.006172,"The Project Gutenberg eBook of The Magna Carta This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under"
5,10019,"Punchinello, Volume 1, No. 22, August 27, 1870",0.005696,"The Project Gutenberg eBook of Punchinello, Volume 1, No. 22, August 27, 1870 This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it,"


Consulta: god heaven angels


Posicion,Identificador,Titulo,Puntuacion,Fragmento
1,10,The King James Version of the Bible,0.032526,"The Project Gutenberg eBook of The King James Version of the Bible This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it awa"
2,1003,"Divine Comedy, Longfellow's Translation, Paradise",0.024944,"The Project Gutenberg eBook of Divine Comedy, Longfellow's Translation, Paradise This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy i"
3,10022,White Queen of the Cannibals: the Story of Mary Slessor of Calabar,0.024217,The Project Gutenberg eBook of White Queen of the Cannibals: the Story of Mary Slessor of Calabar This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoeve
4,10004,The Warriors,0.018900,"The Project Gutenberg eBook of The Warriors This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under th"
5,1004,"Divine Comedy, Longfellow's Translation, Complete",0.017094,"The Project Gutenberg eBook of Divine Comedy, Longfellow's Translation, Complete This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy i"


### Complemento del objetivo BM25

In [20]:
# Complemento del objetivo: BM25
def calcular_bm25(consulta, documentos, k1=1.5, b=0.75):
    numero_documentos = len(documentos)
    if numero_documentos == 0:
        return []
    longitud_media = sum(len(documento) for documento in documentos) / numero_documentos
    frecuencias_df = calcular_df(documentos)
    puntuaciones = []
    for documento in documentos:
        frecuencias_tf = calcular_tf(documento)
        puntuacion = 0.0
        for palabra in set(consulta):
            frecuencia = frecuencias_tf.get(palabra, 0)
            if frecuencia == 0:
                continue
            df = frecuencias_df[palabra]
            idf = math.log(1 + (numero_documentos - df + 0.5) / (df + 0.5))
            ajuste_longitud = 1 - b + b * len(documento) / longitud_media
            puntuacion += idf * frecuencia * (k1 + 1) / (frecuencia + k1 * ajuste_longitud)
        puntuaciones.append(puntuacion)
    return puntuaciones

# Resultado complementario: comparar TF-IDF con BM25
for posicion, consulta in enumerate(consultas):
    puntuaciones_bm25 = calcular_bm25(consultas_procesadas[posicion], documentos)
    ranking_bm25 = construir_ranking(identificadores, titulos, textos_originales, puntuaciones_bm25)
    ranking_tfidf = construir_ranking(identificadores, titulos, textos_originales, similitudes_manuales[posicion])
    print("Consulta:", consulta)
    display(pd.DataFrame({"Posicion": ranking_tfidf["Posicion"], "TF-IDF": ranking_tfidf["Titulo"], "BM25": ranking_bm25["Titulo"]}))

Consulta: love marriage family


Posicion,TF-IDF,BM25
1,"The Works of Aphra Behn, Volume III",The Rivet in Grandfather's Neck: A Comedy of Limitations
2,La Fiammetta,"The Works of Aphra Behn, Volume III"
3,The Complete Works of William Shakespeare,"My First Years as a Frenchwoman, 1876-1879"
4,"Divine Comedy, Longfellow's Translation, Paradise",A Voyage to the Moon
5,The Rivet in Grandfather's Neck: A Comedy of Limitations,The Warriors


Consulta: sea ship whale


Posicion,TF-IDF,BM25
1,"Punchinello, Volume 1, No. 19, August 6, 1870","Punchinello, Volume 1, No. 19, August 6, 1870"
2,Aleutian Indian and English Dictionary,Aleutian Indian and English Dictionary
3,The Mystery,"Wild Northern Scenes; Or, Sporting Adventures with the Rifle and the Rod"
4,Beneath the Banner: Being Narratives of Noble Lives and Brave Deeds,"Punchinello, Volume 1, No. 05, April 30, 1870"
5,The Mountains of California,The Rivet in Grandfather's Neck: A Comedy of Limitations


Consulta: king murder ghost


Posicion,TF-IDF,BM25
1,The Complete Works of William Shakespeare,"The Works of Aphra Behn, Volume III"
2,The King James Version of the Bible,The Complete Works of William Shakespeare
3,"The Works of Aphra Behn, Volume III",The Magnetic North
4,The Magna Carta,"Punchinello, Volume 1, No. 25, September 17, 1870"
5,"Punchinello, Volume 1, No. 22, August 27, 1870",The King James Version of the Bible


Consulta: god heaven angels


Posicion,TF-IDF,BM25
1,The King James Version of the Bible,The Complete Poetical Works of Edgar Allan Poe
2,"Divine Comedy, Longfellow's Translation, Paradise","Divine Comedy, Longfellow's Translation, Paradise"
3,White Queen of the Cannibals: the Story of Mary Slessor of Calabar,"Divine Comedy, Longfellow's Translation, Complete"
4,The Warriors,The King James Version of the Bible
5,"Divine Comedy, Longfellow's Translation, Complete","Divine Comedy, Longfellow's Translation, Purgatory"
